# EDA: помесячная рентабельность (временной ряд)

Запуск: из каталога `backend` активируйте venv и выполните `jupyter notebook` или VS Code; в первой ячейке задайте `PROJECT_ID` и путь к SQLite (или загрузите CSV с колонками `period`, `profitability`, опционально `revenue`, `costs`).

Цели: визуализация ряда, STL (тренд/сезонность/остаток), ACF/PACF, ADF, разбивка train/holdout.

In [ ]:
import os
import sys

# Добавить backend в PYTHONPATH при запуске из notebooks/
BACKEND = os.path.abspath(os.path.join(os.getcwd(), "..", "backend"))
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# Пример: данные из SQLite через SQLAlchemy (async не нужен — синхронный движок)
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

from app.models.metric import Metric
from app.services.forecast.data import metric_df

PROJECT_ID = 1  # <-- замените
# Путь к SQLite: при запуске из notebooks/ по умолчанию ../backend/app.db
_db_default = os.path.abspath(os.path.join(BACKEND, "app.db"))
DB_URL = os.environ.get("FORECAST_EDA_DATABASE_URL", f"sqlite:///{_db_default}")

engine = create_engine(DB_URL)
Session = sessionmaker(bind=engine)
with Session() as session:
    rows = list(session.scalars(select(Metric).where(Metric.project_id == PROJECT_ID).order_by(Metric.year, Metric.month)).all())

df = metric_df(rows)
assert not df.empty, "Нет метрик для проекта — синхронизируйте Sheets или задайте PROJECT_ID"
df = df.sort_values("period")
df.head()

In [ ]:
y = df.set_index("period")["profitability"].asfreq("MS")
fig, ax = plt.subplots(figsize=(11, 4))
y.plot(ax=ax, label="Рентабельность %")
y.rolling(6, min_periods=1).mean().plot(ax=ax, label="MA 6 мес.")
ax.legend()
ax.set_title("Ряд и сглаживание")
plt.show()

if len(y) >= 24:
    stl = STL(y, seasonal=13, robust=True)
    res = stl.fit()
    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
    res.observed.plot(ax=axes[0], title="Исходный")
    res.trend.plot(ax=axes[1], title="Тренд")
    res.seasonal.plot(ax=axes[2], title="Сезонность")
    res.resid.plot(ax=axes[3], title="Остаток")
    plt.tight_layout()
    plt.show()
else:
    print("Для STL желательно ≥24 точек; пропуск STL.")

fig, ax = plt.subplots(figsize=(9, 3))
plot_acf(y.dropna(), lags=min(24, len(y) - 2), ax=ax)
plt.show()
fig, ax = plt.subplots(figsize=(9, 3))
plot_pacf(y.dropna(), lags=min(12, len(y) // 2), ax=ax, method="ywm")
plt.show()

adf_stat, adf_p, *_ = adfuller(y.dropna())
print(f"ADF stat={adf_stat:.4f}, p-value={adf_p:.4f}")

holdout = min(6, max(1, len(y) // 5))
train, val = y.iloc[:-holdout], y.iloc[-holdout:]
fig, ax = plt.subplots(figsize=(11, 4))
train.plot(ax=ax, label="Train")
val.plot(ax=ax, label="Holdout", color="tab:red")
ax.axvline(val.index[0], color="gray", linestyle="--")
ax.legend()
ax.set_title("Разбивка train / validation (последние месяцы)")
plt.show()

## Выводы для отчёта

- Опишите, виден ли устойчивый тренд (наклон STL) и амплитуду сезонности.
- По ACF/PACF: затухающие корреляции — признак стационарности после дифференцирования при необходимости.
- Укажите выбранную схему train/holdout для согласования с backend (`walk_forward_one_step`, последние 6 мес.).